# Singular Value Decomposition (SVD) - Examples

This notebook demonstrates SVD concepts with practical implementations.

## Contents
1. Basic SVD Computation
2. Thin (Economy) SVD
3. SVD from Eigendecomposition
4. Low-Rank Approximation
5. Outer Product Form
6. Matrix Norms via SVD
7. Condition Number
8. Pseudoinverse via SVD
9. Image Compression Simulation
10. PCA via SVD
11. Recommender System (Matrix Factorization)
12. Noise Reduction
13. SVD Geometry Visualization

In [ ]:
import numpy as np
from numpy.linalg import svd, norm, matrix_rank, pinv, eig
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

## 1. Basic SVD Computation

For any matrix $A \in \mathbb{R}^{m \times n}$:
$$A = U \Sigma V^T$$

Where:
- $U$ is $m \times m$ orthogonal (left singular vectors)
- $\Sigma$ is $m \times n$ diagonal (singular values)
- $V$ is $n \times n$ orthogonal (right singular vectors)

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])

print(f"Matrix A (3×2):\n{A}")

# Compute SVD (full)
U, s, Vt = svd(A, full_matrices=True)

print(f"\nFull SVD:")
print(f"U (3×3):\n{np.round(U, 4)}")
print(f"\nSingular values: {np.round(s, 4)}")
print(f"\nVᵀ (2×2):\n{np.round(Vt, 4)}")

In [ ]:
# Reconstruct A from SVD
# Need to create the Σ matrix with proper dimensions
S = np.zeros((3, 2))
S[:2, :2] = np.diag(s)

print(f"Σ (3×2):\n{S}")

reconstructed = U @ S @ Vt
print(f"\nReconstruction U @ Σ @ Vᵀ:\n{np.round(reconstructed, 4)}")
print(f"Matches A: {np.allclose(A, reconstructed)}")

## 2. Thin (Economy) SVD

For tall matrices ($m > n$), the economy SVD avoids computing unused columns of $U$:
- Full: $U_{m \times m}$, $\Sigma_{m \times n}$, $V_{n \times n}$
- Thin: $U_{m \times n}$, $\Sigma_{n \times n}$, $V_{n \times n}$

In [ ]:
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])

print(f"Matrix A (3×2):\n{A}")

# Full SVD
U_full, s, Vt = svd(A, full_matrices=True)

# Thin SVD
U_thin, s_thin, Vt_thin = svd(A, full_matrices=False)

print(f"\nFull SVD: U is {U_full.shape}")
print(f"Thin SVD: U is {U_thin.shape}")

print(f"\nThin U (3×2):\n{np.round(U_thin, 4)}")

# Thin reconstruction
S_thin = np.diag(s_thin)
reconstructed = U_thin @ S_thin @ Vt_thin
print(f"\nThin reconstruction:\n{np.round(reconstructed, 4)}")
print(f"Matches A: {np.allclose(A, reconstructed)}")

## 3. SVD from Eigendecomposition

- Singular values: $\sigma_i = \sqrt{\lambda_i(A^T A)}$
- Right singular vectors $V$: eigenvectors of $A^T A$
- Left singular vectors $U$: $u_i = \frac{1}{\sigma_i} A v_i$

In [ ]:
A = np.array([[3, 1],
              [1, 3]])

print(f"Matrix A:\n{A}")

# Method 1: Direct SVD
U, s, Vt = svd(A)
print(f"\nDirect SVD singular values: {s}")

# Method 2: From A^T A eigendecomposition
AtA = A.T @ A
print(f"\nA^T A:\n{AtA}")

eigenvalues, V = eig(AtA)
# Sort by eigenvalue descending
idx = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[idx]
V = V[:, idx]

print(f"\nEigenvalues of A^T A: {eigenvalues}")
print(f"√eigenvalues (singular values): {np.sqrt(eigenvalues)}")

In [ ]:
# Compute U from U = AV / Σ
singular_values = np.sqrt(eigenvalues)
U_computed = np.zeros_like(A, dtype=float)
for i in range(len(singular_values)):
    U_computed[:, i] = A @ V[:, i] / singular_values[i]

print(f"Computed U from AV/σ:\n{np.round(U_computed, 4)}")
print(f"Direct SVD U:\n{np.round(U, 4)}")
print(f"\n(Sign differences are allowed - eigenvectors are unique up to sign)")

## 4. Low-Rank Approximation

**Eckart-Young Theorem**: The best rank-$k$ approximation is:
$$A_k = \sum_{i=1}^{k} \sigma_i u_i v_i^T$$

Error: $\|A - A_k\|_F = \sqrt{\sum_{i=k+1}^{r} \sigma_i^2}$

In [ ]:
# Create a 4×4 matrix
A = np.array([[1, 2, 3, 4],
              [2, 4, 6, 8],
              [1, 1, 1, 1],
              [3, 3, 3, 3]])

print(f"Original matrix A:\n{A}")
print(f"Rank of A: {matrix_rank(A)}")

U, s, Vt = svd(A)
print(f"\nSingular values: {np.round(s, 4)}")

In [ ]:
# Rank-1 approximation
A1 = s[0] * np.outer(U[:, 0], Vt[0, :])
print(f"Rank-1 approximation A₁:\n{np.round(A1, 2)}")
print(f"Error ||A - A₁||_F: {norm(A - A1, 'fro'):.4f}")

# Rank-2 approximation
A2 = s[0] * np.outer(U[:, 0], Vt[0, :]) + s[1] * np.outer(U[:, 1], Vt[1, :])
print(f"\nRank-2 approximation A₂:\n{np.round(A2, 2)}")
print(f"Error ||A - A₂||_F: {norm(A - A2, 'fro'):.4f}")

# Theoretical error bound
print(f"\nTheoretical error √(σ₃² + σ₄²) = {np.sqrt(s[2]**2 + s[3]**2):.4f}")

## 5. Outer Product Form

$$A = \sum_{i=1}^{r} \sigma_i u_i v_i^T$$

Each term is a rank-1 matrix. This representation shows how SVD decomposes $A$ into layers of decreasing importance.

In [ ]:
A = np.array([[4, 0],
              [3, -5]])

print(f"Matrix A:\n{A}")

U, s, Vt = svd(A)

print(f"\nSingular values: σ₁={s[0]:.4f}, σ₂={s[1]:.4f}")
print(f"U:\n{np.round(U, 4)}")
print(f"Vᵀ:\n{np.round(Vt, 4)}")

In [ ]:
# Outer product form: A = σ₁u₁v₁ᵀ + σ₂u₂v₂ᵀ
term1 = s[0] * np.outer(U[:, 0], Vt[0, :])
term2 = s[1] * np.outer(U[:, 1], Vt[1, :])

print(f"σ₁u₁v₁ᵀ =\n{np.round(term1, 4)}")
print(f"\nσ₂u₂v₂ᵀ =\n{np.round(term2, 4)}")

print(f"\nSum σ₁u₁v₁ᵀ + σ₂u₂v₂ᵀ:\n{np.round(term1 + term2, 4)}")
print(f"Matches A: {np.allclose(A, term1 + term2)}")

## 6. Matrix Norms via SVD

| Norm | Formula | Meaning |
|------|---------|--------|
| Spectral | $\|A\|_2 = \sigma_1$ | Maximum stretch factor |
| Frobenius | $\|A\|_F = \sqrt{\sum \sigma_i^2}$ | Total "energy" |
| Nuclear | $\|A\|_* = \sum \sigma_i$ | Sum of stretches |

In [ ]:
A = np.array([[1, 2],
              [3, 4]])

print(f"Matrix A:\n{A}")

U, s, Vt = svd(A)
print(f"\nSingular values: {s}")

# Spectral norm (2-norm) = σ₁
spectral_norm = s[0]
print(f"\nSpectral norm ||A||₂ = σ₁ = {spectral_norm:.4f}")
print(f"NumPy norm(A, 2): {norm(A, 2):.4f}")

# Frobenius norm = √(Σσᵢ²)
frobenius_svd = np.sqrt(np.sum(s**2))
print(f"\nFrobenius norm ||A||_F = √(Σσᵢ²) = {frobenius_svd:.4f}")
print(f"NumPy norm(A, 'fro'): {norm(A, 'fro'):.4f}")

# Nuclear norm = Σσᵢ
nuclear = np.sum(s)
print(f"\nNuclear norm ||A||_* = Σσᵢ = {nuclear:.4f}")
print(f"NumPy nuclear norm: {np.linalg.norm(A, 'nuc'):.4f}")

## 7. Condition Number

$$\kappa(A) = \frac{\sigma_1}{\sigma_r}$$

- $\kappa \approx 1$: Well-conditioned (stable)
- $\kappa \gg 1$: Ill-conditioned (numerically unstable)

In [ ]:
# Well-conditioned matrix
A = np.array([[2, 1],
              [1, 2]])

U, s, Vt = svd(A)
cond_A = s[0] / s[-1]

print("Well-conditioned matrix:")
print(f"A:\n{A}")
print(f"Singular values: {s}")
print(f"Condition number κ(A) = σ₁/σᵣ = {cond_A:.4f}")

In [ ]:
# Ill-conditioned matrix
B = np.array([[1, 1],
              [1, 1.0001]])

U, s, Vt = svd(B)
cond_B = s[0] / s[-1]

print("Ill-conditioned matrix:")
print(f"B:\n{B}")
print(f"Singular values: {s}")
print(f"Condition number κ(B) = {cond_B:.1f}")

print("\nEffect on solving Ax = b:")
print(f"Well-conditioned: small errors in b → small errors in x")
print(f"Ill-conditioned: errors amplified by up to {cond_B:.1f}×")

## 8. Pseudoinverse via SVD

The Moore-Penrose pseudoinverse:
$$A^+ = V \Sigma^+ U^T$$

where $\Sigma^+$ inverts non-zero singular values: $(\Sigma^+)_{ii} = 1/\sigma_i$ if $\sigma_i > 0$, else $0$.

In [ ]:
# Overdetermined system (more rows than columns)
A = np.array([[1, 2],
              [3, 4],
              [5, 6]])

print(f"Matrix A (3×2):\n{A}")

U, s, Vt = svd(A)

# Compute pseudoinverse: A⁺ = V Σ⁺ Uᵀ
S_pinv = np.zeros((2, 3))
for i in range(len(s)):
    if s[i] > 1e-10:
        S_pinv[i, i] = 1 / s[i]

A_pinv = Vt.T @ S_pinv @ U.T

print(f"\nPseudoinverse A⁺ (via SVD):\n{np.round(A_pinv, 4)}")
print(f"NumPy pinv:\n{np.round(pinv(A), 4)}")

In [ ]:
# Verify pseudoinverse properties
print("Verification of pseudoinverse properties:")
print(f"AA⁺A = A: {np.allclose(A @ A_pinv @ A, A)}")
print(f"A⁺AA⁺ = A⁺: {np.allclose(A_pinv @ A @ A_pinv, A_pinv)}")

# Solving least squares
b = np.array([1, 2, 3])
x_lstsq = A_pinv @ b
print(f"\nLeast squares solution for Ax ≈ b = {b}:")
print(f"x = A⁺b = {x_lstsq}")
print(f"Residual ||Ax - b||: {norm(A @ x_lstsq - b):.4f}")

## 9. Image Compression Simulation

SVD enables lossy compression by keeping only top $k$ singular values:
- Original: $mn$ values
- Compressed: $k(m + n + 1)$ values

In [ ]:
# Create a sample "image" matrix (structured, not random)
np.random.seed(42)
m, n = 50, 50

x = np.linspace(0, 4*np.pi, m)
y = np.linspace(0, 4*np.pi, n)
X, Y = np.meshgrid(x, y)
image = np.sin(X) * np.cos(Y) + 0.1 * np.random.randn(m, n)

print(f"Original 'image' size: {m}×{n} = {m*n} values")

# SVD
U, s, Vt = svd(image)
print(f"\nSingular values (first 10): {np.round(s[:10], 4)}")

In [ ]:
# Compression at different ranks
ranks = [1, 5, 10, 20, 50]

print("Compression results:")
print(f"{'Rank k':>8} {'Storage':>12} {'Compression':>12} {'Error':>12}")

for k in ranks:
    # Reconstruct with rank k
    image_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    
    # Storage: k columns of U + k values + k rows of Vt
    storage = k * (m + n + 1)
    compression = 100 * (1 - storage / (m * n))
    error = norm(image - image_k, 'fro') / norm(image, 'fro')
    
    print(f"{k:8d} {storage:12d} {compression:11.1f}% {error:12.4f}")

In [ ]:
# Visualize compression quality
fig, axes = plt.subplots(2, 3, figsize=(12, 8))

axes[0, 0].imshow(image, cmap='viridis')
axes[0, 0].set_title('Original')
axes[0, 0].axis('off')

for idx, k in enumerate([1, 5, 10, 20, 50]):
    ax = axes[(idx+1)//3, (idx+1)%3]
    image_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
    ax.imshow(image_k, cmap='viridis')
    error = 100 * norm(image - image_k, 'fro') / norm(image, 'fro')
    ax.set_title(f'Rank {k} ({error:.1f}% error)')
    ax.axis('off')

plt.tight_layout()
plt.show()

## 10. PCA via SVD

For centered data matrix $X$:
- Principal components = columns of $V$ (right singular vectors)
- Variance explained = $\sigma_i^2 / (n-1)$
- Projections = $X V_k$

In [ ]:
# Generate correlated 2D data
np.random.seed(42)
n = 100

# Create correlated data
t = np.random.randn(n)
X = np.column_stack([
    2*t + 0.1*np.random.randn(n),
    t + 0.1*np.random.randn(n)
])

print(f"Data shape: {X.shape}")

# Center the data
X_centered = X - X.mean(axis=0)

# SVD of centered data
U, s, Vt = svd(X_centered, full_matrices=False)

print(f"\nSingular values: {s}")
print(f"\nPrincipal components (rows of Vᵀ):\n{Vt}")

In [ ]:
# Variance explained
variance_explained = s**2 / (n - 1)
total_variance = variance_explained.sum()

print(f"Variance explained by each PC:")
for i, (var, pct) in enumerate(zip(variance_explained, 
                                    100 * variance_explained / total_variance)):
    print(f"  PC{i+1}: {var:.4f} ({pct:.1f}%)")

# Visualization
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_centered[:, 0], X_centered[:, 1], alpha=0.5)

# Plot principal components
origin = [0, 0]
for i in range(2):
    scale = np.sqrt(variance_explained[i]) * 2
    pc = Vt[i, :] * scale
    color = 'red' if i == 0 else 'blue'
    ax.annotate('', xy=pc, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=color, lw=3))
    ax.annotate(f'PC{i+1}', pc + 0.1, fontsize=12, color=color)

ax.set_xlabel('X₁')
ax.set_ylabel('X₂')
ax.set_title('PCA via SVD')
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
plt.show()

## 11. Recommender System (Matrix Factorization)

SVD can factorize a user-item rating matrix to discover latent factors:
- Users are mapped to a latent space
- Items are mapped to the same space
- Predicted rating = user vector · item vector

In [ ]:
# User-Item rating matrix (0 = missing)
# 4 users, 5 items
R = np.array([
    [5, 3, 0, 1, 4],
    [4, 0, 0, 1, 0],
    [1, 1, 0, 5, 4],
    [0, 1, 5, 4, 0]
], dtype=float)

print("Rating matrix (0 = missing):")
print(R)

# Fill missing with row means (simple approach)
R_filled = R.copy()
for i in range(R.shape[0]):
    row_mean = R[i, R[i] > 0].mean()
    R_filled[i, R[i] == 0] = row_mean

print(f"\nFilled matrix:\n{np.round(R_filled, 2)}")

In [ ]:
# SVD
U, s, Vt = svd(R_filled)
print(f"Singular values: {np.round(s, 4)}")

# Low-rank approximation (k=2 latent factors)
k = 2
R_approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

print(f"\nRank-{k} approximation:")
print(np.round(R_approx, 2))

# Predictions for missing entries
print("\nPredicted ratings for originally missing entries:")
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        if R[i, j] == 0:
            print(f"  User {i+1}, Item {j+1}: {R_approx[i, j]:.2f}")

## 12. Noise Reduction

High singular values capture signal; low singular values capture noise.
Truncating to keep only top $k$ components removes noise.

In [ ]:
# Create clean signal (low-rank)
np.random.seed(42)
m, n = 20, 20

# Low-rank clean matrix (rank 2)
u1 = np.sin(np.linspace(0, np.pi, m))
v1 = np.cos(np.linspace(0, np.pi, n))
u2 = np.cos(np.linspace(0, 2*np.pi, m))
v2 = np.sin(np.linspace(0, 2*np.pi, n))

clean = 5 * np.outer(u1, v1) + 3 * np.outer(u2, v2)

# Add noise
noise = 0.5 * np.random.randn(m, n)
noisy = clean + noise

print(f"Clean matrix rank: {matrix_rank(clean)}")
print(f"Noisy matrix rank: {matrix_rank(noisy)}")

In [ ]:
# SVD of noisy data
U, s, Vt = svd(noisy)

print(f"Singular values (first 8): {np.round(s[:8], 4)}")
print("Note: First 2 are much larger (signal), rest is noise")

# Denoise by keeping top k components
k = 2
denoised = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]

print(f"\nError comparison:")
print(f"  ||noisy - clean||_F = {norm(noisy - clean, 'fro'):.4f}")
print(f"  ||denoised - clean||_F = {norm(denoised - clean, 'fro'):.4f}")
improvement = 100 * (1 - norm(denoised - clean, 'fro') / norm(noisy - clean, 'fro'))
print(f"  Improvement: {improvement:.1f}%")

In [ ]:
# Visualize denoising
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(clean, cmap='viridis')
axes[0].set_title('Clean Signal')
axes[0].axis('off')

axes[1].imshow(noisy, cmap='viridis')
axes[1].set_title('Noisy')
axes[1].axis('off')

axes[2].imshow(denoised, cmap='viridis')
axes[2].set_title(f'Denoised (k={k})')
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 13. SVD Geometry Visualization

SVD shows that any linear transformation is:
1. Rotation (by $V^T$)
2. Scaling (by $\Sigma$)
3. Rotation (by $U$)

In [ ]:
A = np.array([[3, 1],
              [1, 3]])

U, s, Vt = svd(A)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Generate unit circle
theta = np.linspace(0, 2*np.pi, 100)
circle = np.array([np.cos(theta), np.sin(theta)])

# Step 0: Original circle
ax = axes[0]
ax.plot(circle[0], circle[1], 'b-', linewidth=2)
ax.set_title('1. Unit Circle')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# Step 1: After Vᵀ (rotation)
after_Vt = Vt @ circle
ax = axes[1]
ax.plot(after_Vt[0], after_Vt[1], 'g-', linewidth=2)
ax.set_title('2. After Vᵀ (rotation)')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# Step 2: After Σ (scaling)
S = np.diag(s)
after_S = S @ after_Vt
ax = axes[2]
ax.plot(after_S[0], after_S[1], 'orange', linewidth=2)
ax.set_title(f'3. After Σ (scale by {s[0]:.2f}, {s[1]:.2f})')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

# Step 3: After U (rotation)
after_U = U @ after_S
ax = axes[3]
ax.plot(after_U[0], after_U[1], 'r-', linewidth=2)
ax.set_title('4. After U (rotation) = A × circle')
ax.set_xlim(-4, 4)
ax.set_ylim(-4, 4)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

## Summary

| Concept | Formula | NumPy |
|---------|---------|-------|
| SVD | $A = U\Sigma V^T$ | `np.linalg.svd(A)` |
| Singular values from $A^TA$ | $\sigma_i = \sqrt{\lambda_i(A^TA)}$ | eigenvalues |
| Low-rank approximation | $A_k = \sum_{i=1}^k \sigma_i u_i v_i^T$ | truncate |
| Pseudoinverse | $A^+ = V\Sigma^+U^T$ | `np.linalg.pinv(A)` |
| Spectral norm | $\|A\|_2 = \sigma_1$ | `norm(A, 2)` |
| Frobenius norm | $\|A\|_F = \sqrt{\sum \sigma_i^2}$ | `norm(A, 'fro')` |
| Condition number | $\kappa(A) = \sigma_1/\sigma_r$ | `np.linalg.cond(A)` |

### ML Applications
- **PCA**: Principal components from right singular vectors
- **Recommender systems**: Matrix factorization for collaborative filtering
- **Image compression**: Low-rank approximation
- **Noise reduction**: Truncation removes noise
- **Latent Semantic Analysis**: Document similarity